# Detection & Analysis of Pre-SN Outburst Simulations

[View our paper here](https://iopscience.iop.org/article/10.3847/1538-4357/ad973d) for details on our cleaning and binning method, as well as our pre-SN outburst detection process!

## Settings 
### Load the config file and preset column names

In [ ]:
# ==== User Settings ====

# Enter the config file name:
CONFIG_FILE = "config.ini"

# Enter the column name preset to use 
# (see config file for more; default options: "atlas", "rubin", "tess"):
PRESET = "tess" # "atlas"

# =======================

import sys, json
import numpy as np
import pandas as pd
from typing import Optional, List
import matplotlib.pyplot as plt

from download import load_config, make_dir_if_not_exists
from lightcurve import PresetColumnNames, Supernova, AveragedSupernova, LightCurve, SimDetecLightCurve, SimDetecSupernova
from plot import Plot, PlotLimits
from utils import load_json_config, load_preset_column_names_from_config, reformat_dir, get_config_flags, get_inverse_mjd_ranges, hexstring_to_int, format_float_string
from step1_generate_sim_tables import get_model_settings, parse_config_params, parse_colname_info, SimTables, get_sim_tables_output_dir, Param, Params, ListParam, RangeParam, LogRangeParam, RandomParam, RandomInRangeParam, ParamType
from step2_generate_detec_tables import AtlasInjectionLoop, TessInjectionLoop, get_detec_tables_output_dir, Gaussian, AsymmetricGaussian, Model
from step3_calculate_efficiencies import ContaminationTable, EfficiencyTable, MagnitudeThresholdTable, AtlasAnalysisLoop

# load config
config = load_config(CONFIG_FILE)

# load preset
colnames = load_preset_column_names_from_config(PRESET, config)

### Load the light curves

In [ ]:
# ==== User Settings ====

# Enter the target SN name:
TNSNAME = '2019vxm'

# Enter the path to the data directory that contains the ATClean-readable SN directories:
INPUT_DIR = config["dir"]["atclean_input"]

# Enter the path to the directory that contains cleaned and binned light curves, plots, and more:
OUTPUT_DIR = config["dir"]["output"]

# Automatically save the generated plots in this notebook?:
SAVE_PLOTS = False
# Enter the path to a directory to optionally save plots:
PLOTS_DIR = f'{OUTPUT_DIR}/{TNSNAME}/bump_analysis/plots'

# Enter the filter for this light curve:
FILT = 'tess' #'o'

# Enter the SN's discovery date:
DISCOVERY_DATE = 58798.04 #58650.0#

# Enter the size of the MJD bins (in days):
MJD_BIN_SIZE = 0.125 #1.0

# Enter the number of control light curves to load:
NUM_CONTROLS = int(config["download"]["num_controls"]) #7

# Edit the default color scheme here:
COLOR_SCHEME = {
    "sn_flux": {
        # ATLAS
        "o": "orange",
        "c": "cyan",
        # Rubin
        "u": "purple",
        "g": "green",
        "r": "salmon",
        "i": "indigo",
        "z": "brown",
        "y": "darkred",
        # TESS
        "tess": "plum",
    },
    "sn_flagged_flux": "red",
    "control_flux": "steelblue",
    "select_control_flux": "forestgreen",
    "face": "whitesmoke",
    "sn_fom": "deeppink",
    "control_fom": "cornflowerblue",
    "select_control_fom": "mediumblue",
    "sim_object_flux": "darkgreen"
}

# =======================

# make sure tnsname not attached to end of input and output dirs
INPUT_DIR = reformat_dir(INPUT_DIR, TNSNAME)
OUTPUT_DIR = reformat_dir(OUTPUT_DIR, TNSNAME)

badday_flag = hexstring_to_int(config['averaging']['flag'])
print(f"SN {TNSNAME}, filter '{FILT}', {NUM_CONTROLS} control light curves, Bad Day Cut flag {hex(badday_flag)}, discovery date: {DISCOVERY_DATE}")
print(f'Output directory (load cleaned and binned light curves): {OUTPUT_DIR}/{TNSNAME}')
print(f'Plots directory (save plots): {PLOTS_DIR}')

# load cleaned light curves
sn = Supernova(colnames, tnsname=TNSNAME, mjd0=DISCOVERY_DATE, filt=FILT)
sn.load_all(OUTPUT_DIR, NUM_CONTROLS, cleaned=True)

# load binned light curves
avg_sn = SimDetecSupernova(
    colnames, 
    tnsname=TNSNAME, 
    mjdbinsize=MJD_BIN_SIZE, 
    mjd0=DISCOVERY_DATE, 
    filt=FILT, 
    flag=badday_flag
)
avg_sn.load_all(OUTPUT_DIR, NUM_CONTROLS)

p = Plot(output_dir=PLOTS_DIR, color_scheme=COLOR_SCHEME)

config_flags = get_config_flags(config)

## Plot cleaned original and binned pre-SN light curves

Note that if there are no pre-MJD0 measurements, the entire light curve will be plotted.

In [ ]:
# ==== User Settings ====

# Optionally, manually enter the x and y limits for the plot:
XLIM_LOWER = None
XLIM_UPPER = None
YLIM_LOWER = None
YLIM_UPPER = None

# =======================

custom_lims = PlotLimits(
    xlower=XLIM_LOWER,
    xupper=XLIM_UPPER,
    ylower=YLIM_LOWER,
    yupper=YLIM_UPPER,
)

fig = p.plot_preSN(
    sn, 
    avg_sn, 
    config_flags,
    custom_lims=custom_lims,
    save=SAVE_PLOTS
)

## Plot all control light curves

In [ ]:
# ==== User Settings ====

# Plot the SN light curve as well?
INCLUDE_SN = False

# Use two columns instead of one? 
# (only works if there are an even number of light curves)
TWO_COLUMNS = (avg_sn.num_controls + int(INCLUDE_SN)) % 2 == 0

# Optionally, manually enter the x and y limits for the plot:
XLIM_LOWER = None
XLIM_UPPER = None
YLIM_LOWER = None
YLIM_UPPER = None

# =======================

custom_lims = PlotLimits(
    xlower=XLIM_LOWER,
    xupper=XLIM_UPPER,
    ylower=YLIM_LOWER,
    yupper=YLIM_UPPER,
)

fig = p.plot_all_controls(
    avg_sn, 
    config_flags,
    custom_lims=custom_lims,
    two_columns=TWO_COLUMNS,
    include_sn=INCLUDE_SN,
    save=SAVE_PLOTS
)

## Plot binned pre-SN and control light curve

In [ ]:
# ==== User Settings ====

# Enter the index of the control light curve to plot:
SELECT_CONTROL_INDEX = 2

# Optionally, manually enter the x and y limits for the plot:
XLIM_LOWER = None
XLIM_UPPER = None
YLIM_LOWER = None
YLIM_UPPER = None

# =======================

custom_lims = PlotLimits(
    xlower=XLIM_LOWER,
    xupper=XLIM_UPPER,
    ylower=YLIM_LOWER,
    yupper=YLIM_UPPER,
)

fig = p.plot_binned_examples(
    avg_sn, 
    SELECT_CONTROL_INDEX, 
    custom_lims=custom_lims, 
    save=SAVE_PLOTS
)

## Get detection limits w.r.t. contamination

We use the bisection method to calculate the best detection limits $\Sigma_{\text{FOM, limit}}$ for each $\sigma_{\text{kernel}}$ with respect to a target contamination value.

We measure contamination for a given $\sigma_{\text{kernel}}$ and its corresponding $\Sigma_{\text{FOM, limit}}$ by counting the number of false positives in the control light curves (where a single false positive is defined by a continuous sequence of $\Sigma_{\text{FOM}}$ above $\Sigma_{\text{FOM, limit}}$). We record two alternative measures of contamination for evaluating a certain $\Sigma_{\text{FOM, limit}}$:

1. the number of positive control light curves (default option)
2. the total number of false positives throughout all control light curve

#### Plot observation seasons

In [ ]:
# ==== User Settings ====

# List the observation seasons 
# (set to empty list [] for the entire span of the light curve):
MJD_RANGES: List[List[float]] = [
    [58765.9375, 58775.1875],
    [58777.1875, 58786.6875],
    [58792.0625, 58801.4375],
    [58804.0625, 58814.5625]
    # [57233.5, 57328.5], 
    # [57466.5, 57535.5], 
    # [57914.5, 58122.5],
    # [58206.5, 58486.5],
    # [58594.5, 58836.5],
    # [58910.5, 59212.5],
    # [59304.5, 59564.5],
    # [59637.5, 59915.5],
    # [60023.5, 60309.5],
    # [60387.5, 60671.5],
    # [60727.5, 60770.5]
]

# Optionally, manually enter the x and y limits for the plot:
XLIM_LOWER = None
XLIM_UPPER = None
YLIM_LOWER = None
YLIM_UPPER = None

# =======================

custom_lims = PlotLimits(
    xlower=XLIM_LOWER,
    xupper=XLIM_UPPER,
    ylower=YLIM_LOWER,
    yupper=YLIM_UPPER,
)

if MJD_RANGES is None or len(MJD_RANGES) < 1:
    min_mjd, max_mjd = avg_sn.lcs[0].get_min_and_max_mjd()
    print(f"Setting MJD ranges to (min MJD {min_mjd}, max MJD {max_mjd})")
    MJD_RANGES = [[min_mjd, max_mjd]]

fig = p.plot_mjd_ranges(
    sn, 
    avg_sn, 
    config_flags,
    MJD_RANGES,
    suptitle="Observation Seasons",
    custom_lims=custom_lims,
    filename="observation_seasons",
    save=SAVE_PLOTS
)

### Exclude bad MJD ranges and determine valid observation seasons

In [ ]:
# ==== User Settings ====

# Optionally expand observation gap MJD ranges by +-exclude_obs_edges days on either side 
# (set to None or 0 to skip):
EXPAND_EDGES = MJD_BIN_SIZE * 3

# Optionally exclude additional bad observation seasons or arbitrary ranges 
# (set to None or empty list [] to skip):
EXCLUDE_MJD_RANGES: List[List[float]] = [
    # [57233.0, TEMPLATE_CHANGE_1_MJD],
    # [TEMPLATE_CHANGE_2_MJD, 60771.0]
]

# Optionally, manually enter the x and y limits for the plot:
XLIM_LOWER = None
XLIM_UPPER = None
YLIM_LOWER = None
YLIM_UPPER = None

# =======================

min_mjd, max_mjd = avg_sn.lcs[0].get_min_and_max_mjd()
print(f'Min MJD: {min_mjd}, max MJD: {max_mjd}')

print(f'Observation season ranges: {MJD_RANGES}')

excluded_mjd_ranges = get_inverse_mjd_ranges(
    MJD_RANGES, 
    min_mjd, 
    max_mjd, 
    expand_edges=EXPAND_EDGES,
    exclude_mjd_ranges=EXCLUDE_MJD_RANGES
)
print(f'Excluded MJD ranges (observation season gaps plus expanded edges and additional excluded ranges): \n\t{excluded_mjd_ranges}')

included_mjd_ranges = get_inverse_mjd_ranges(
    excluded_mjd_ranges, 
    min_mjd, 
    max_mjd
)
print(f"Included ranges: \n\t{included_mjd_ranges}")

custom_lims = PlotLimits(
    xlower=XLIM_LOWER,
    xupper=XLIM_UPPER,
    ylower=YLIM_LOWER,
    yupper=YLIM_UPPER,
)

if len(excluded_mjd_ranges) > 0:
    fig = p.plot_mjd_ranges(
        sn, 
        avg_sn, 
        config_flags,
        excluded_mjd_ranges,
        suptitle="Excluded MJD ranges",
        range_color="tomato",
        custom_lims=custom_lims,
        filename="excluded_mjd_ranges",
        save=SAVE_PLOTS
    )

if len(included_mjd_ranges) > 0:
    fig = p.plot_mjd_ranges(
        sn, 
        avg_sn, 
        config_flags,
        included_mjd_ranges,
        suptitle="Included MJD ranges",
        range_color="forestgreen",
        custom_lims=custom_lims,
        filename="included_mjd_ranges",
        save=SAVE_PLOTS
    )

### Calculate the preliminary range of valid detection limits $\Sigma_{\text{FOM, limit}}$

As our starting range of valid detection limits $\Sigma_{\text{FOM, limit}}$ for each $\sigma_{\text{kernel}}$, we set our lower limit to 0 and our upper limit to the maximum $\Sigma_{\text{FOM}}$ value across all control light curves.

In [ ]:
# ==== User Settings ====

# Enter a list of possible sigma kernels (in days) of the weighted gaussian rolling sum:
SIGMA_KERNS: List[float] = LogRangeParam(
    "sigma_kerns",
    minval=0.125, #1
    maxval=1, #3
    base=10,
    n=5,
).values

# Exclude certain control lcs from the analysis 
# (set to None or empty list [] for no exclusions):
SKIP_CONTROL_INDICES = []

# Optionally, manually enter the x and y limits for the plot:
XLIM_LOWER = None
XLIM_UPPER = None
YLIM_LOWER = None
YLIM_UPPER = None

# =======================

print(f"Skipping control light curves: {SKIP_CONTROL_INDICES}")
avg_sn.remove_lc_indices(SKIP_CONTROL_INDICES)

print(f'\nsigma_kerns: {SIGMA_KERNS}\n')
avg_sn.set_mjd_ranges(included_mjd_ranges)
all_fom_dict, prelim_fom_limit_ranges = avg_sn.get_prelim_fom_limit_ranges(SIGMA_KERNS)

### Calculate contamination for preliminary detection limits $\Sigma_{\text{FOM, limit}}$

In [ ]:
contam = ContaminationTable()
contam.construct_prelim_t(
    avg_sn, 
    SIGMA_KERNS, 
    prelim_fom_limit_ranges
)
print(contam)

### Plot $\Sigma_{\text{FOM}}$ and preliminary detection limits $\Sigma_{\text{FOM, limit}}$ for all $\sigma_{\text{kernel}}$

In [ ]:
# ==== User Settings ====

# Enter the index of the control light curve to highlight:
SELECT_CONTROL_INDEX = 2

# =======================

fig = p.plot_fom(
    avg_sn, 
    all_fom_dict,
    SIGMA_KERNS, 
    SELECT_CONTROL_INDEX, 
    fom_limits=prelim_fom_limit_ranges.get_all_single(),
    save=SAVE_PLOTS
)

### Calculate the best detection limits $\Sigma_{\text{FOM, limit}}$

We use the starting range of valid detection limits $\Sigma_{\text{FOM, limit}}$ and their corresponding contamination values to calculate the detection limit with the ideal contamination. Below, we allow the user to select the desired measure of contamination (i.e., the number of positive control light curves). 

We iterate for n_steps steps. If we have a starting range of detection limits $\Sigma_{\text{FOM, limit}}$ from $a$ to $b$ and iterate for $n$ steps, we obtain the detection limit with the desired contamination with an accuracy of $\frac{b-a}{2^n}$.

In [ ]:
# ==== User Settings ====

# Enter the target contamination
# (i.e. the target number of positive control light curves for a given FOM limit):
TARGET_VALUE = 2

# Enter the maximum number of iterations for refining FOM limits:
N_STEPS = 15

# Load an existing ContaminationTable from file instead of calculating it?
LOAD = False

# =======================

detec_tables_dir = get_detec_tables_output_dir(OUTPUT_DIR, TNSNAME)

if LOAD:
    contam.load(detec_tables_dir)
    print(contam)
    fom_limits = contam.get_fom_limits_from_t()
else:
    fom_limits = contam.calculate(
        avg_sn, 
        prelim_fom_limit_ranges, 
        SIGMA_KERNS,
        target_value=TARGET_VALUE,
        n_steps=N_STEPS,
        verbose=True
    )
    contam.save(detec_tables_dir)
# print(contam)
print(f"Best FOM limits for sigma_kerns: {fom_limits}")

### Plot the best detection limits $\Sigma_{\text{FOM, limit}}$ and scan the SN light curve for detections

In [ ]:
fig = p.plot_fom_dists(
    SIGMA_KERNS, 
    all_fom_dict, 
    fom_limits.get_all_single(), 
    xlim_scale=1.4
)
fig = p.plot_fom(
    avg_sn, 
    all_fom_dict,
    SIGMA_KERNS, 
    SELECT_CONTROL_INDEX,
    fom_limits=fom_limits.get_all_single(),
    save=SAVE_PLOTS
)

print()
avg_sn.scan_sn_for_detections(SIGMA_KERNS, fom_limits)

## Generate simulation tables using a predefined model and its parameters

In [ ]:
# ==== User Settings ====

# Enter the name of the file with default parameter settings for possible simulation models:
SIM_SETTINGS_FILENAME = "step1_settings.json"

# Enter the name of the model in SIM_SETTINGS_FILENAME to simulate:
MODEL = "gaussian"

# =======================

sim_config = load_json_config(SIM_SETTINGS_FILENAME)
model_settings = get_model_settings(MODEL, sim_config)

filename, mjd_colname, mag_colname, flux_colname = parse_colname_info(
    model_settings, MODEL
)

### Option 1: Use a settings file like `simulation_settings.json`

In [ ]:
print("Parsing model parameters...")
params = parse_config_params(
    model_settings,
    time_param_name=model_settings["time_parameter_name"],
    brightness_param_name=model_settings["brightness_parameter_name"],
)
params.validate()

### Option 2: Manually create `Param` objects

Create your parameter objects here, then add them to the `params` collection. Possible parameter objects include `ListParam`, `RangeParam`, `LogRangeParam`, `RandomParam`, `RandomInRangeParam`. We have provided the code for Gaussian simulations below, as an example.

Remember to set `param_type=ParamType.BRIGHTNESS` for your peak apparent magnitude parameter, and `param_type=ParamType.TIME` for your time parameter.

Tip: If you'd like to add an unchangeable/constant parameter that will not be varied, but should be there for informational purposes, you can create a ListParam with a list of one entry. For example:
```py
# Create a parameter called "sigma_sim" that will always have a value of 2.8:
sigma_sim_param = ListParam("sigma_sim", [2.8])
# Add it to the Params object:
params.add(sigma_sim_param)
# Now the generated SimTables will have a sigma_sim column populated entirely with 2.8.
```

In [ ]:
params = Params()

peak_appmag_param = RangeParam(
    "peak_appmag",
    minval=16.0,
    maxval=23.0,
    step=0.5,
    param_type=ParamType.BRIGHTNESS
)
params.add(peak_appmag_param)

sigma_sim_param = LogRangeParam(
    "sigma_sim",
    minval=0.125,
    maxval=1,
    base=10,
    n=5,
    n_digits=5,
)
params.add(sigma_sim_param)

peak_mjd_param = RandomInRangeParam(
    "peak_mjd",
    valid_ranges=included_mjd_ranges,
    n=10,
    param_type=ParamType.TIME
)
params.add(peak_mjd_param)

params.validate()
print("\n",params)

### Generate simulation tables using the loaded *or* created `Param`s

In [ ]:
sim_tables = SimTables(MODEL)
sim_tables.generate(
    params,
    filename=filename,
    mjd_colname=mjd_colname,
    mag_colname=mag_colname,
    flux_colname=flux_colname,
)
sim_tables_dir = get_sim_tables_output_dir(OUTPUT_DIR, TNSNAME)
sim_tables.save_all(sim_tables_dir)

## Inject simulations into control light curves

In [ ]:
# ==== User Settings ====

# Load existing SimDetecTables (True), or generate new ones from existing SimTables (False)?
LOAD = False

# =======================

# injection_loop = AtlasInjectionLoop(SIGMA_KERNS, MODEL, sim_tables_dir, detec_tables_dir)
injection_loop = TessInjectionLoop(SIGMA_KERNS, MODEL, sim_tables_dir, detec_tables_dir)
injection_loop.set_sn(avg_sn)
print()

if LOAD:
    injection_loop.get_brightness_param_from_detec_tables()
    injection_loop.load_detec_tables()
else:
    injection_loop.get_brightness_param_from_sim_tables()
    injection_loop.load_sim_tables()
    injection_loop.loop()

### Plot injection of example Simulation

In [ ]:
# ==== User Settings ====

# Enter the kernel size of the rolling sum:
SIGMA_KERN = SIGMA_KERNS[0]

# Enter the brightness of the simulated object:
BRIGHTNESS = params.brightness_param.values[len(params.brightness_param.values) // 2]

# Enter the index of the control light curve into which we will inject the Simulation:
CONTROL_INDEX = SELECT_CONTROL_INDEX

# Enter the MJD at which the Simulation will reach peak brightness:
PEAK_MJD = 58780.0

# Optionally, manually enter the MJD window 
# so that the x limits of the plot are PEAK_MJD±MJD_WINDOW:
MJD_WINDOW = 40

# Optionally, manually enter the flux window 
# so that the flux y limits of the plot are 0±FLUX_WINDOW:
FLUX_WINDOW = 60

# Optionally, manually enter the FOM window 
# so that the FOM y limits of the plot are 0±FOM_WINDOW:
FOM_WINDOW = 8

# =======================

print(f"Applying rolling sum with sigma_kern {format_float_string(SIGMA_KERN)}")
injection_loop._sn.lcs[CONTROL_INDEX].apply_rolling_sum(SIGMA_KERN, flag=injection_loop._sn.flag)

print(f"Injecting simulation with brightness {format_float_string(BRIGHTNESS)} and peak MJD {format_float_string(PEAK_MJD)} into control light curve #{CONTROL_INDEX}")

custom_lims = PlotLimits(
    xlower=XLIM_LOWER,
    xupper=XLIM_UPPER,
    ylower=YLIM_LOWER,
    yupper=YLIM_UPPER,
)

# Example simulation: Gaussian
sim = Gaussian()
sim_flux, sim_lc = injection_loop.add_simulation_to_lc(
    SIGMA_KERN, 
    BRIGHTNESS,
    CONTROL_INDEX,
    sim,
    sigma_sim=sigma_sim_param.values[0],
    time_peak_mjd=PEAK_MJD
) 
fig = p.plot_simulated_lc(
    sim_lc, 
    sim_flux,
    SIGMA_KERN, 
    fom_limits.get(SIGMA_KERN),
    PEAK_MJD, 
    flag=injection_loop._sn.flag,
    time_window=MJD_WINDOW,
    flux_window=FLUX_WINDOW,
    fom_window=FOM_WINDOW,
    save=SAVE_PLOTS,
    filename="sim_gauss"
)

# Example simulation: Model
sim = Model(
    filename="./extern/eruption_m3e-07.dat",
    mjd_colname=None,
    mag_colname=None,
    flux_colname=False,
    model_name="charlie_model",
)
sim_flux, sim_lc = injection_loop.add_simulation_to_lc(
    SIGMA_KERN, 
    BRIGHTNESS,
    CONTROL_INDEX,
    sim,
    time_peak_mjd=PEAK_MJD
)
fig = p.plot_simulated_lc(
    sim_lc, 
    sim_flux,
    SIGMA_KERN, 
    fom_limits.get(SIGMA_KERN),
    PEAK_MJD, 
    flag=injection_loop._sn.flag,
    time_window=MJD_WINDOW,
    flux_window=FLUX_WINDOW,
    fom_window=FOM_WINDOW,
    save=SAVE_PLOTS,
    filename="sim_model"
)

## Calculate & plot efficiencies

In [ ]:
# ==== User Settings ====

SELECT_PARAM_NAME = sigma_sim_param.name

# =======================

effs = EfficiencyTable(SIGMA_KERNS, params)
effs.calculate_efficiencies(injection_loop.tables, fom_limits)
effs.save(detec_tables_dir, MODEL)

for sigma_kern in effs.sigma_kerns:
    fig = p.plot_efficiency(sigma_kern, effs, SELECT_PARAM_NAME, save=SAVE_PLOTS)

## Calculate & plot magnitude thresholds for 50% and 80% efficiency

This section computes and visualizes the magnitude detection thresholds corresponding to different detection efficiency targets (e.g., 50% and 80%).

In [ ]:
# ==== User Settings ====

#  Enter the efficiency levels (in %) for which we want to find magnitude thresholds:
PERCENTS = [50.0, 80.0]

# Enter whether to use a logarithmic scale for the x-axis in the plot:
LOG = False

# Enter the base of the logarithmic scale (will be used if LOG=True):
BASE = 10

# =======================

mt = MagnitudeThresholdTable()
mt.calculate(effs, SELECT_PARAM_NAME, percents=PERCENTS)
fig = p.plot_all_mag_thresholds(mt, log=LOG, base=BASE, save=SAVE_PLOTS)